## Step 1 Imports

In [1]:
import numpy as np
from collections import defaultdict

In [2]:
a = ['a','b','c','d','e']
for i, c in enumerate(a):
    print(f"{i},{c}")

0,a
1,b
2,c
3,d
4,e


## Step 2 Data

In [3]:
sentences = [
    "I love dogs",
    "I adore puppies",
    "dogs are great pets",
    "puppies are cute animals",
    "the stock market crashed",
    "financial markets are volatile",
    "stocks and bonds are investments",
]

## Step 3 Build Vocab

In [4]:
words = [w for s in sentences for w in s.lower().split()]
print(words)
vocab = sorted(set(words))
print(f"{len(vocab)}\n{vocab}")

w2i = {w:i for i,w in enumerate(vocab)}
print(w2i.items())
i2w = {i:w for w,i in w2i.items()}
print(i2w)
V = len(vocab)


['i', 'love', 'dogs', 'i', 'adore', 'puppies', 'dogs', 'are', 'great', 'pets', 'puppies', 'are', 'cute', 'animals', 'the', 'stock', 'market', 'crashed', 'financial', 'markets', 'are', 'volatile', 'stocks', 'and', 'bonds', 'are', 'investments']
21
['adore', 'and', 'animals', 'are', 'bonds', 'crashed', 'cute', 'dogs', 'financial', 'great', 'i', 'investments', 'love', 'market', 'markets', 'pets', 'puppies', 'stock', 'stocks', 'the', 'volatile']
dict_items([('adore', 0), ('and', 1), ('animals', 2), ('are', 3), ('bonds', 4), ('crashed', 5), ('cute', 6), ('dogs', 7), ('financial', 8), ('great', 9), ('i', 10), ('investments', 11), ('love', 12), ('market', 13), ('markets', 14), ('pets', 15), ('puppies', 16), ('stock', 17), ('stocks', 18), ('the', 19), ('volatile', 20)])
{0: 'adore', 1: 'and', 2: 'animals', 3: 'are', 4: 'bonds', 5: 'crashed', 6: 'cute', 7: 'dogs', 8: 'financial', 9: 'great', 10: 'i', 11: 'investments', 12: 'love', 13: 'market', 14: 'markets', 15: 'pets', 16: 'puppies', 17: 'sto

## Step 4 Generate Training Pairs

In [5]:
def get_training_pairs(sentences, window = 2):
    pairs = []
    for sentence in sentences:
        tokens = sentence.lower().split()
        for i, center in enumerate(tokens):
            for j in range(max(0,i-window),min(len(tokens),i+window+1)):
                if i!=j:
                    pairs.append((w2i[center],w2i[tokens[j]]))
    return pairs

pairs = get_training_pairs(sentences)
print(pairs)
print(f"Training pairs: {len(pairs)}")
print(f"Sample pairs: {[(i2w[c], i2w[t]) for c,t in pairs[:5]]}\n")


[(10, 12), (10, 7), (12, 10), (12, 7), (7, 10), (7, 12), (10, 0), (10, 16), (0, 10), (0, 16), (16, 10), (16, 0), (7, 3), (7, 9), (3, 7), (3, 9), (3, 15), (9, 7), (9, 3), (9, 15), (15, 3), (15, 9), (16, 3), (16, 6), (3, 16), (3, 6), (3, 2), (6, 16), (6, 3), (6, 2), (2, 3), (2, 6), (19, 17), (19, 13), (17, 19), (17, 13), (17, 5), (13, 19), (13, 17), (13, 5), (5, 17), (5, 13), (8, 14), (8, 3), (14, 8), (14, 3), (14, 20), (3, 8), (3, 14), (3, 20), (20, 14), (20, 3), (18, 1), (18, 4), (1, 18), (1, 4), (1, 3), (4, 18), (4, 1), (4, 3), (4, 11), (3, 1), (3, 4), (3, 11), (11, 4), (11, 3)]
Training pairs: 66
Sample pairs: [('i', 'love'), ('i', 'dogs'), ('love', 'i'), ('love', 'dogs'), ('dogs', 'i')]



## Step 5 Network

In [6]:
EMBED_DIM = 5

W1 = np.random.randn(V, EMBED_DIM) * 0.01
W2 = np.random.randn(EMBED_DIM, V) * 0.01 

print(W1)
print(W2)

# print(W1.shape)
# r, c = W1.shape

# for i in range(c):
#     W1[10][i] = 1

# r2, c2 = W2.shape
# for i in range(r2):
#     for j in range(c2):
#         W2[i][j] = 1

# print(W2)
    
def softmax(x):
    e = np.exp(x - x.max())   # subtract max for numerical stability
    return e / e.sum()

[[ 0.01619485 -0.0101544  -0.0082737   0.00177839  0.01756109]
 [ 0.0043468   0.01318918 -0.00893906  0.01340802  0.00223752]
 [ 0.0021079  -0.01390224 -0.00347669  0.00961438 -0.01758218]
 [-0.00056442 -0.00168498 -0.0050738   0.00847529  0.00718147]
 [ 0.02213438 -0.00427418 -0.0003479  -0.00539553  0.00686746]
 [-0.00787444 -0.00444882 -0.00352227 -0.00511654  0.00550096]
 [ 0.01568149  0.00015102  0.0105416  -0.00634067  0.00505746]
 [-0.00497826  0.01708761 -0.00812015  0.02513956 -0.01166116]
 [ 0.00349391 -0.00028454  0.00770903  0.00350914  0.01085184]
 [-0.01815189  0.00858229  0.00780645  0.01183304 -0.00037184]
 [ 0.00910299 -0.00240317 -0.00311588  0.0048585  -0.00555104]
 [-0.01133112 -0.00303596  0.00895979 -0.00150214 -0.0036937 ]
 [ 0.01065696  0.01961969 -0.00430407  0.01741496 -0.00921552]
 [ 0.01505787  0.01117175  0.00504161  0.00595329 -0.00059947]
 [-0.0187313  -0.00983181 -0.01700993 -0.00892252  0.01194807]
 [-0.00120548  0.01763467 -0.0088987   0.00311306 -0.00

## Step 6 Training

In [7]:
print(pairs[0])
center_idx = pairs[0][0]
print(center_idx)
h = W1[center_idx]
print(h)
scores = h @ W2
print(scores)

(10, 12)
10
[ 0.00910299 -0.00240317 -0.00311588  0.0048585  -0.00555104]
[-3.56928047e-05 -5.68679206e-05 -1.84359459e-04  3.18810714e-05
 -1.48638992e-04  1.97763452e-04  5.18493066e-05  1.90580869e-05
 -1.02263768e-04  3.72635546e-05 -8.29865964e-05 -2.14558418e-04
  7.84803638e-05  1.06151313e-05  1.98803408e-04 -4.13910983e-05
  1.14457590e-05 -1.10075597e-04  1.89171846e-04 -1.45370278e-04
 -1.14518363e-05]


In [8]:
lr = 0.05
losses = []

for epoch in range(200):
    total_loss = 0
    for center_idx, target_idx in pairs:
        # print(i2w[center_idx], i2w[target_idx])
        # ── Forward pass ──
        h = W1[center_idx]                  # just a row lookup — no matrix multiply needed
        scores = h @ W2                     # (EMBED_DIM,) @ (EMBED_DIM, V) → (V,)
        probs = softmax(scores)             # probability over whole vocab

        # ── Loss (cross-entropy) ──
        loss = -np.log(probs[target_idx] + 1e-9)
        total_loss += loss

        # ── Backward pass ──
        dscores = probs.copy()
        dscores[target_idx] -= 1            # gradient of softmax + cross-entropy

        dW2 = np.outer(h, dscores)          # (EMBED_DIM, V)
        dh  = W2 @ dscores                  # (EMBED_DIM,)

        # ── Update weights ──
        W2 -= lr * dW2
        W1[center_idx] -= lr * dh          # only update the row we used

    losses.append(total_loss / len(pairs))
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d}  loss: {total_loss/len(pairs):.4f}")
print(f"Epoch {epoch:3d}  loss: {total_loss/len(pairs):.4f}")

Epoch   0  loss: 3.0445
Epoch  50  loss: 2.2376
Epoch 100  loss: 1.6723
Epoch 150  loss: 1.6091
Epoch 199  loss: 1.6018


In [9]:
# ── 6. Extract embeddings — W1 is what we keep ───────────
embeddings = W1
print(f"\nEmbedding for 'dogs':   {embeddings[w2i['dogs']].round(3)}")
print(f"Embedding for 'puppies': {embeddings[w2i['puppies']].round(3)}")




Embedding for 'dogs':   [ 0.663  1.893 -1.27   0.761 -0.405]
Embedding for 'puppies': [ 0.233  0.342  0.216  2.255 -1.671]


In [10]:
# ── 7. Find most similar words ───────────────────────────
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def most_similar(word, top_n=3):
    vec = embeddings[w2i[word]]
    scores = [(i2w[i], cosine_sim(vec, embeddings[i]))
              for i in range(V) if i2w[i] != word]
    return sorted(scores, key=lambda x: -x[1])[:top_n]

print()
for word in ['dogs', 'stocks', 'adore']:
    similar = most_similar(word)
    print(f"Most similar to '{word}': {similar}")


Most similar to 'dogs': [('pets', np.float64(0.8657980149126727)), ('love', np.float64(0.6229682700044799)), ('and', np.float64(0.5093535434852063))]
Most similar to 'stocks': [('are', np.float64(0.7291027512335945)), ('investments', np.float64(0.414656839072565)), ('bonds', np.float64(0.3244537578096752))]
Most similar to 'adore': [('love', np.float64(0.7072039694005764)), ('i', np.float64(0.5894797111127322)), ('cute', np.float64(0.4816761125754629))]


# Use pre-trained embeddings
